# GPU ACCELERATION

In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

# IMPORTS

In [6]:
import os
import pandas as pd
from playwright.async_api import async_playwright

# MAIN DATASET

In [3]:
df = pd.read_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv')

# URLs TO SCRAPE

In [4]:
evo3_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_three-stage_evolution'
evo2_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_two-stage_evolution'

# IMPLEMENT PLAYWRIGHT SCRAPING AND DATA CLEANING

## SCRAPING

In [8]:
DATA_DIR = os.path.expanduser('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/')


async def get_pokes_rows_pw(page, url):
    """Navigate to url and return all non-header rows from .prettytable tables."""
    await page.goto(url, wait_until='domcontentloaded')
    await page.wait_for_selector('.mw-parser-output .prettytable')
    rows = []
    for table in await page.locator('.mw-parser-output .prettytable').all():
        for i, row in enumerate(await table.locator('tr').all()):
            if i > 0:
                rows.append(row)
    return rows


async def get_name(cell):
    """Return the Pokémon name from a table cell (second anchor if present, else first, else cell text)."""
    anchors = await cell.locator('a').all()
    if len(anchors) >= 2:
        return await anchors[1].inner_text()
    elif len(anchors) == 1:
        return await anchors[0].inner_text()
    return await cell.inner_text()


async def pokes_parser1_pw(row):
    """Parse a three-stage evolution table row → dict with ev1/ev2/ev3 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 3:
        return {'ev1': await get_name(cells[0]), 'ev2': await get_name(cells[1]), 'ev3': await get_name(cells[2])}
    elif n == 2:
        return {'ev2': await get_name(cells[0]), 'ev3': await get_name(cells[1])}
    else:
        return {'ev3': await get_name(cells[0])}


async def pokes_parser2_pw(row):
    """Parse a two-stage evolution table row → dict with ev1/ev2 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 2:
        first_anchors = await cells[0].locator('a').all()
        ev1 = await first_anchors[1].inner_text() if len(first_anchors) >= 2 else await cells[0].inner_text()
        ev2 = await get_name(cells[1])
        return {'ev1': ev1, 'ev2': ev2}
    else:
        cell_anchors = await cells[0].locator('a').all()
        ev2 = await cell_anchors[1].inner_text() if len(cell_anchors) >= 2 else await cells[0].inner_text()
        return {'ev2': ev2}


async def main():
    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=True)
        page = await browser.new_page()

        rows3 = await get_pokes_rows_pw(page, evo3_url)
        evo3_data = [await pokes_parser1_pw(r) for r in rows3]
        pd.DataFrame(evo3_data).to_csv(DATA_DIR + 'evo3.csv', index=False)

        rows2 = await get_pokes_rows_pw(page, evo2_url)
        evo2_data = [await pokes_parser2_pw(r) for r in rows2]
        pd.DataFrame(evo2_data).to_csv(DATA_DIR + 'evo2.csv', index=False)

        await browser.close()

await main()
print("Scraping complete. CSVs saved to data directory.")

Scraping complete. CSVs saved to data directory.


## CLEANING

In [9]:
evo3 = pd.read_csv(DATA_DIR + 'evo3.csv')
evo2 = pd.read_csv(DATA_DIR + 'evo2.csv')

# Clean evo2: fill NaN ev1 values and strip extra text after newlines
evo2['ev1'] = evo2['ev1'].fillna('none')
for i in range(len(evo2) - 1):
    evo2.at[i, 'ev1'] = str(evo2.iloc[i]['ev1']).split('\n')[0]
    evo2.at[i, 'ev2'] = str(evo2.iloc[i]['ev2']).split('\n')[0]

# Merge three-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo3['ev1'])
df['ev2'] = df['Pokemon'].isin(evo3['ev2'])
df['ev3'] = df['Pokemon'].isin(evo3['ev3'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, True]
df.loc[df['ev3'] == True, ['Evolution Stage', 'Evolves']] = [3, False]

df.drop(['ev1', 'ev2', 'ev3'], axis=1, inplace=True)

# Merge two-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo2['ev1'])
df['ev2'] = df['Pokemon'].isin(evo2['ev2'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, False]

df.drop(['ev1', 'ev2'], axis=1, inplace=True)

# Fill remaining NaN evolution stages (non-evolving Pokémon)
df.loc[df['Evolution Stage'].isna(), ['Evolution Stage', 'Evolves']] = [0, False]

print("Cleaning and merging complete.")

Cleaning and merging complete.


# RENAMING DATAFRAME FOR THE FUN OF IT AND SAVING NEW DATA TO THE `pokemon_dataset_MASTER.csv` FILE FOR SAFE KEEPING

In [11]:
pokes = df

In [13]:
pokes.to_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv')

# CREATE COLUMNS OF RATIOS BETWEEN THE DIFFERENT STATS

In [14]:
pokes['Attack/Defense'] = pokes['Attack'] / pokes['Defense']
pokes['Special Attack/Special Defense'] = pokes['Special Attack'] / pokes['Special Defense']
pokes['Attack/Special Attack'] = pokes['Attack'] / pokes['Special Attack']
pokes['Defense/Special Defense'] = pokes['Defense'] / pokes['Special Defense']
pokes['Speed/Defense'] = pokes['Speed'] / pokes['Defense']
pokes['Speed/Attack'] = pokes['Speed'] / pokes['Attack']
pokes['HP/Defense'] = pokes['HP'] / pokes['Defense']
pokes['HP/Special Defense'] = pokes['HP'] / pokes['Special Defense']

# SAVE TO THE `,csv` FILE AGAIN (FREQUENT SAVING ENSURES WE DON'T SCREWUP BEYOND FIXING---in my stupid opinion anyhow

In [15]:
pokes.to_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv')